In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

In [2]:
spark = SparkSession.builder \
    .appName("NYC_Taxi_Data_Pipeline") \
    .getOrCreate()

C:\Users\Murch24\aws-pyspark-data-engineering-project\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [6]:
df = spark.read.parquet("../data/raw/yellow_tripdata_2014-03.parquet")

In [7]:
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: integer (nullable = true)
 |-- airport_fee: integer (nullable = true)



In [8]:
df.show(5)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2014-03-01 00:55:41|  2014-03-01 00:57:42|              1|          0.0|         1|                 N|         145|         145|           2|        3.0|  0.5|    0.5|       0.

In [9]:
df.count()

15428134

In [10]:
len(df.columns)

19

In [15]:
df.select(
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "store_and_fwd_flag"
).describe().show()

+-------+------------------+------------------+------------------+------------------+--------------------+-----------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      total_amount|congestion_surcharge|airport_fee|store_and_fwd_flag|
+-------+------------------+------------------+------------------+------------------+--------------------+-----------+------------------+
|  count|          15428134|          15428134|          15428134|          15428134|                   0|          0|           7620823|
|   mean|3.8134811766614884|12.215581225830778|1.4731240686662246|14.761783681786595|                NULL|       NULL|              NULL|
| stddev|1982.6178988411455|10.092548583388256| 2.247891680206382|12.243391462701169|                NULL|       NULL|              NULL|
|    min|               0.0|           -612.42|               0.0|               0.0|                NULL|       NULL|                 N|
|    max|         5005013.0|      

In [17]:
df.select(
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "store_and_fwd_flag"
).summary().show()

+-------+------------------+------------------+------------------+------------------+--------------------+-----------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      total_amount|congestion_surcharge|airport_fee|store_and_fwd_flag|
+-------+------------------+------------------+------------------+------------------+--------------------+-----------+------------------+
|  count|          15428134|          15428134|          15428134|          15428134|                   0|          0|           7620823|
|   mean|3.8134811766614884|12.215581225830778|1.4731240686662246|14.761783681786595|                NULL|       NULL|              NULL|
| stddev|1982.6178988411455|10.092548583388256| 2.247891680206382|12.243391462701169|                NULL|       NULL|              NULL|
|    min|               0.0|           -612.42|               0.0|               0.0|                NULL|       NULL|                 N|
|    25%|              1.01|      

In [18]:
null_counts = df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
)

null_counts.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|              0|            0|         0|           7807311|           0|           0|           0|          0|    0|      0|         

## Observations

- The dataset contains 15,428,134 taxi trips.
- There are 19 columns.
- Pickup and dropoff timestamps are stored as `timestamp_ntz`.
- `congestion_surcharge` and `airport_fee` contain many NULL values because these fees were not applicable to older trips.
- The dataset appears suitable for further transformation and analysis.

In [19]:
df.filter(col("trip_distance") < 0).count()

0

In [20]:
df.filter(col("total_amount") < 0).count()

0

In [22]:
from pyspark.sql.functions import min, max

df.select(
    min("tpep_pickup_datetime").alias("min_pickup"),
    max("tpep_pickup_datetime").alias("max_pickup")
).show()

+-------------------+-------------------+
|         min_pickup|         max_pickup|
+-------------------+-------------------+
|2014-03-01 00:00:00|2014-03-31 23:59:58|
+-------------------+-------------------+



## Data Quality Check Observations

- The dataset contains 15,428,134 taxi trips for March 2014.
- No negative values were found in `trip_distance` and `total_amount`.
- Pickup timestamps range from 2014-03-01 to 2014-03-31.
- `congestion_surcharge` and `airport_fee` contain NULL values for all records because these fields were not applicable for this historical dataset.
- `store_and_fwd_flag` contains approximately 7.8M NULL values and may require handling during transformation.

In [4]:
import pandas as pd
print(pd.__version__)

3.0.5


In [1]:
import pandas as pd
print(pd.__version__)

2.3.3
